In [ ]:
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize, closing, disk

# Imports directos de vuestro proyecto (PyCharm los lee solos)
from src.utils import get_xml_path, parse_stenosis_xml, im_show, load_image, get_all_image_paths, group_paths_by_serie
from src.preprocessing import preprocess_bs_nlm_clahe
from src.vessel_segmentation import apply_cleaned_hysteresis, apply_frangi, _sample_patient_frames

In [ ]:
paths = get_all_image_paths()
series = group_paths_by_serie(paths)
print(f"Total series found: {len(series)}")

# Parámetros fijos que ya teníais calculados
KERNEL_SIZE = 81
H = 8
CLIP_LIMIT = 4.0
SCALE_MIN = 4
SCALE_MAX = 15
LOW_THRESH = 0.02
HIGH_THRESH = 0.18
MIN_SIZE = 200

SEED = random.randint(1, 999)
loaded = _sample_patient_frames(series, n=9, seed=716)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.measure import label

# Diccionario para guardar el esqueleto completo y los datos necesarios de cada paciente
resultados_esqueletos = {}

# Bucle principal para procesar y guardar todos los esqueletos
for pid, _ in loaded:
    serie_paths = next(v for k, v in series.items() if k.split('/')[0] == pid)
    path = serie_paths[len(serie_paths) // 2]
    img = load_image(path)
    boxes = parse_stenosis_xml(get_xml_path(path))

    if not boxes:
        continue

    # 1. Pipeline de vuestro grupo
    proc = preprocess_bs_nlm_clahe(img, kernel_size=KERNEL_SIZE, h=H, clip_limit=CLIP_LIMIT)
    frangi_map = apply_frangi(proc, scale_min=SCALE_MIN, scale_max=SCALE_MAX, black_ridges=True)
    bool_mask = apply_cleaned_hysteresis(frangi_map, low_thresh=LOW_THRESH, high_thresh=HIGH_THRESH, min_size=MIN_SIZE)

    # 2. Esqueletización directa
    skeleton_raw = skeletonize(bool_mask)

    # =========================================================================
    # 3. QUEDARSE CON TODOS LOS SKELETONS (Sin filtros ni descartes)
    # =========================================================================
    # Hacemos una copia directa para conservar absolutamente todas las ramas encontradas
    skeleton = skeleton_raw.copy()

    # =========================================================================
    # Guardamos todo lo necesario para el Montecarlo en el diccionario
    # =========================================================================
    resultados_esqueletos[pid] = {
        'skeleton': skeleton,
        'proc': proc,
        'img_shape': img.shape,
        'boxes': boxes
    }

    # 4. Visualización (Imagen preprocesada + Todos los fragmentos en azul)
    visual_skel = cv2.cvtColor(proc, cv2.COLOR_GRAY2BGR)

    # Pintamos TODOS los fragmentos del esqueleto en AZUL
    visual_skel[skeleton] = [255, 0, 0]

    # Dibujamos la caja de la estenosis en ROJO
    for box in boxes:
        cv2.rectangle(
            visual_skel,
            (box['xmin'], box['ymin']),
            (box['xmax'], box['ymax']),
            color=(0, 0, 255),
            thickness=2
        )

    im_show(visual_skel, title=f'All Skeletons Maintained — Patient: {pid}', figsize=(8, 8))

In [ ]:
import random
import cv2
import numpy as np

# Configuración del Montecarlo Adaptativo
ROI_SIZE = 80
N_MONTECARLO = 100  # <--- Cambiado a 100 ventanas fijas
half_size = ROI_SIZE // 2

# Recorremos el diccionario con los datos ya precalculados de cada paciente
for pid, datos in resultados_esqueletos.items():
    skeleton = datos['skeleton']
    proc = datos['proc']
    img_shape = datos['img_shape']
    boxes = datos['boxes']

    # Extraer coordenadas del esqueleto aprobado
    y_coords, x_coords = np.where(skeleton)
    skeleton_points = list(zip(x_coords, y_coords))

    total_puntos = len(skeleton_points)
    montecarlo_boxes = []

    if total_puntos > 0:
        # ESTRATEGIA: Seleccionar 100 índices distribuidos uniformemente (equidistantes)
        indices_equidistantes = np.linspace(0, total_puntos - 1, N_MONTECARLO, dtype=int)

        for idx in indices_equidistantes:
            pt_x, pt_y = skeleton_points[idx]

            mc_xmin = int(pt_x - half_size)
            mc_ymin = int(pt_y - half_size)
            mc_xmax = int(pt_x + half_size)
            mc_ymax = int(pt_y + half_size)

            # Si la ventana se sale por los bordes, la recolocamos (la empujamos hacia dentro)
            # para no perder la muestra y asegurar las 100 ventanas siempre.
            if mc_xmin < 0:
                mc_xmax -= mc_xmin
                mc_xmin = 0
            if mc_ymin < 0:
                mc_ymax -= mc_ymin
                mc_ymin = 0
            if mc_xmax >= img_shape[1]:
                mc_xmin -= (mc_xmax - img_shape[1] + 1)
                mc_xmax = img_shape[1] - 1
            if mc_ymax >= img_shape[0]:
                mc_ymin -= (mc_ymax - img_shape[0] + 1)
                mc_ymax = img_shape[0] - 1

            montecarlo_boxes.append({'xmin': mc_xmin, 'ymin': mc_ymin, 'xmax': mc_xmax, 'ymax': mc_ymax})

    # Construcción de la Imagen y Ploteo
    visual_skel = cv2.cvtColor(proc, cv2.COLOR_GRAY2BGR)

    # Pintamos el esqueleto azul
    visual_skel[skeleton] = [255, 0, 0]

    # 1. Crear y dibujar la NUEVA ventana de Ground Truth centrada de tamaño ROI_SIZE (Mantiene el dibujo rojo)
    for box in boxes:
        center_x = (box['xmin'] + box['xmax']) // 2
        center_y = (box['ymin'] + box['ymax']) // 2

        gt_fixed = {
            'xmin': int(center_x - half_size),
            'ymin': int(center_y - half_size),
            'xmax': int(center_x + half_size),
            'ymax': int(center_y + half_size)
        }

        cv2.rectangle(
            visual_skel,
            (gt_fixed['xmin'], gt_fixed['ymin']),
            (gt_fixed['xmax'], gt_fixed['ymax']),
            color=(0, 0, 255), # Rojo en BGR
            thickness=2
        )

    # 2. Analizar y dibujar las ventanas de Montecarlo (Comparando contra la BOX ORIGINAL ROSA)
    for mc_box in montecarlo_boxes:
        pintar_rojo = False

        # El solapamiento ahora se calcula contra la caja original del XML
        for box in boxes:
            inter_xmin = max(mc_box['xmin'], box['xmin'])
            inter_ymin = max(mc_box['ymin'], box['ymin'])
            inter_xmax = min(mc_box['xmax'], box['xmax'])
            inter_ymax = min(mc_box['ymax'], box['ymax'])

            inter_w = max(0, inter_xmax - inter_xmin)
            inter_h = max(0, inter_ymax - inter_ymin)
            area_interseccion = inter_w * inter_h

            area_box_original = (box['xmax'] - box['xmin']) * (box['ymax'] - box['ymin'])

            # Condición: ¿Contiene como mínimo el 40% del área de la caja original XML?
            if area_box_original > 0 and area_interseccion >= 0.40 * area_box_original:
                pintar_rojo = True
                break

        color_box = (0, 0, 255) if pintar_rojo else (0, 255, 0)

        cv2.rectangle(
            visual_skel,
            (mc_box['xmin'], mc_box['ymin']),
            (mc_box['xmax'], mc_box['ymax']),
            color=color_box,
            thickness=1
        )

    # 3. Dibujamos la caja ORIGINAL de la estenosis (XML) en ROSA
    for box in boxes:
        cv2.rectangle(
            visual_skel,
            (box['xmin'], box['ymin']),
            (box['xmax'], box['ymax']),
            color=(255, 0, 255), # Rosa/Magenta en BGR
            thickness=1
        )

    im_show(visual_skel, title=f'Equidistant Montecarlo ROIs — Patient: {pid}', figsize=(8, 8))
    print(f"Paciente {pid} -> Ventanas Montecarlo generadas: {len(montecarlo_boxes)}")

In [ ]:
import random
import cv2
import numpy as np

# Configuración del Montecarlo
ROI_SIZE = 80
N_MONTECARLO = 30
RADIO_EXCLUSION = 70
half_size = ROI_SIZE // 2

# Recorremos el diccionario con los datos ya precalculados de cada paciente
for pid, datos in resultados_esqueletos.items():
    skeleton = datos['skeleton']
    proc = datos['proc']
    img_shape = datos['img_shape']
    boxes = datos['boxes']

    # Extraer coordenadas del esqueleto aprobado
    y_coords, x_coords = np.where(skeleton)
    skeleton_points = list(zip(x_coords, y_coords))

    montecarlo_boxes = []
    centros_guardados = []

    # Mezclamos los puntos al azar para este paciente
    random.seed(random.randint(0, 100))
    random.shuffle(skeleton_points)

    # Bucle de muestreo
    for pt_x, pt_y in skeleton_points:
        if len(montecarlo_boxes) >= N_MONTECARLO:
            break

        mc_xmin = int(pt_x - half_size)
        mc_ymin = int(pt_y - half_size)
        mc_xmax = int(pt_x + half_size)
        mc_ymax = int(pt_y + half_size)

        if mc_xmin < 0 or mc_ymin < 0 or mc_xmax >= img_shape[1] or mc_ymax >= img_shape[0]:
            continue

        demasiado_cerca = False
        for cx, cy in centros_guardados:
            distancia = np.sqrt((pt_x - cx)**2 + (pt_y - cy)**2)
            if distancia < RADIO_EXCLUSION:
                demasiado_cerca = True
                break

        if not demasiado_cerca:
            montecarlo_boxes.append({'xmin': mc_xmin, 'ymin': mc_ymin, 'xmax': mc_xmax, 'ymax': mc_ymax})
            centros_guardados.append((pt_x, pt_y))

    # Construcción de la Imagen y Ploteo
    visual_skel = cv2.cvtColor(proc, cv2.COLOR_GRAY2BGR)

    # Pintamos el esqueleto azul
    visual_skel[skeleton] = [255, 0, 0]

    # 1. Crear y dibujar la NUEVA ventana de Ground Truth centrada de tamaño ROI_SIZE (Mantiene el dibujo rojo)
    for box in boxes:
        center_x = (box['xmin'] + box['xmax']) // 2
        center_y = (box['ymin'] + box['ymax']) // 2

        gt_fixed = {
            'xmin': int(center_x - half_size),
            'ymin': int(center_y - half_size),
            'xmax': int(center_x + half_size),
            'ymax': int(center_y + half_size)
        }

        cv2.rectangle(
            visual_skel,
            (gt_fixed['xmin'], gt_fixed['ymin']),
            (gt_fixed['xmax'], gt_fixed['ymax']),
            color=(0, 0, 255), # Rojo en BGR
            thickness=2
        )

    # 2. Analizar y dibujar las ventanas de Montecarlo (Comparando contra la BOX ORIGINAL ROSA)
    for mc_box in montecarlo_boxes:
        pintar_rojo = False

        # El solapamiento ahora se calcula contra la caja original del XML
        for box in boxes:
            # Coordenadas de la intersección entre Montecarlo y la caja XML original
            inter_xmin = max(mc_box['xmin'], box['xmin'])
            inter_ymin = max(mc_box['ymin'], box['ymin'])
            inter_xmax = min(mc_box['xmax'], box['xmax'])
            inter_ymax = min(mc_box['ymax'], box['ymax'])

            # Calculamos ancho y alto de la intersección
            inter_w = max(0, inter_xmax - inter_xmin)
            inter_h = max(0, inter_ymax - inter_ymin)
            area_interseccion = inter_w * inter_h

            # El área total de la ventana Ground Truth original (Rosa)
            area_box_original = (box['xmax'] - box['xmin']) * (box['ymax'] - box['ymin'])

            # Condición: ¿Contiene como mínimo el 40% del área de la caja original XML?
            if area_box_original > 0 and area_interseccion >= 0.40 * area_box_original:
                pintar_rojo = True
                break

        # Asignamos el color definitivo (Rojo si cumple, si no Verde)
        color_box = (0, 0, 255) if pintar_rojo else (0, 255, 0)

        cv2.rectangle(
            visual_skel,
            (mc_box['xmin'], mc_box['ymin']),
            (mc_box['xmax'], mc_box['ymax']),
            color=color_box,
            thickness=1
        )

    # 3. Dibujamos la caja ORIGINAL de la estenosis (XML) en ROSA
    for box in boxes:
        cv2.rectangle(
            visual_skel,
            (box['xmin'], box['ymin']),
            (box['xmax'], box['ymax']),
            color=(255, 0, 255), # Rosa/Magenta en BGR
            thickness=1
        )

    im_show(visual_skel, title=f'Montecarlo Spaced ROIs — Patient: {pid}', figsize=(8, 8))
    print(f"Paciente {pid} -> Ventanas Montecarlo generadas: {len(montecarlo_boxes)}")